## Cell 1 — Imports
We import all libraries upfront. This is clean coding practice — interviewers notice when imports are scattered.

In [1]:
import pandas as pd
import numpy as np
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ Libraries imported successfully")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

✅ Libraries imported successfully
   pandas  : 3.0.1
   numpy   : 2.4.2


In [4]:
# ⚠️  CHANGE THIS to your actual folder
BASE_DIR = "C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics"

RAW_DIR     = os.path.join(BASE_DIR, "data", "raw")
CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
EXPORTS_DIR = os.path.join(BASE_DIR, "exports")

os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(EXPORTS_DIR, exist_ok=True)

print("✅ Paths configured")
print(f"   Raw data → {RAW_DIR}")
print(f"   Exports  → {EXPORTS_DIR}")

✅ Paths configured
   Raw data → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\data\raw
   Exports  → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\exports


In [5]:
year_files = sorted(glob.glob(os.path.join(RAW_DIR, "amazon_india_20*.csv")))

print(f"Found {len(year_files)} year files:")
all_years = []
for filepath in year_files:
    year   = int(re.search(r'(\d{4})', os.path.basename(filepath)).group(1))
    temp   = pd.read_csv(filepath, low_memory=False)
    temp['source_year'] = year
    all_years.append(temp)
    print(f"   {year}: {len(temp):,} rows")

df_raw = pd.concat(all_years, ignore_index=True)
print(f"\n✅ Combined: {len(df_raw):,} rows × {df_raw.shape[1]} columns")

Found 11 year files:
   2015: 33,165 rows
   2016: 55,275 rows
   2017: 77,385 rows
   2018: 99,495 rows
   2019: 121,605 rows
   2020: 143,715 rows
   2021: 138,187 rows
   2022: 132,660 rows
   2023: 127,132 rows
   2024: 121,605 rows
   2025: 77,385 rows

✅ Combined: 1,127,609 rows × 35 columns


In [6]:
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("\n--- Data Types ---")
print(df_raw.dtypes.to_string())

print("\n--- Missing Values ---")
nulls     = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(1)
report    = pd.DataFrame({'missing_count': nulls, 'missing_%': nulls_pct})
print(report[report['missing_count'] > 0].to_string())

df_raw.head(3)

Shape: 1,127,609 rows × 35 columns

--- Data Types ---
transaction_id                str
order_date                    str
customer_id                   str
product_id                    str
product_name                  str
category                      str
subcategory                   str
brand                         str
original_price_inr            str
discount_percent          float64
discounted_price_inr      float64
quantity                    int64
subtotal_inr              float64
delivery_charges          float64
final_amount_inr          float64
customer_city                 str
customer_state                str
customer_tier                 str
customer_spending_tier        str
customer_age_group            str
payment_method                str
delivery_days                 str
delivery_type                 str
is_prime_member               str
is_festival_sale              str
festival_name                 str
customer_rating               str
return_status              

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,discounted_price_inr,quantity,subtotal_inr,delivery_charges,final_amount_inr,customer_city,customer_state,customer_tier,customer_spending_tier,customer_age_group,payment_method,delivery_days,delivery_type,is_prime_member,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating,source_year
0,TXN_2015_00000001,2015-01-25,CUST_2015_00003884,PROD_000021,Samsung Galaxy S6 16GB Black,Electronics,Smartphones,Samsung,123614.29,27.91,"89,112.17",1,"89,112.17",0.00,"89,112.17",Mumbai,Maharashtra,Metro,Premium,46-55,COD,6,Standard,No,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.19,True,4.70,2015
1,TXN_2015_00000002,2015-01-05,CUST_2015_00011709,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,"54,731.86",1,"54,731.86",0.00,"54,731.86",Allahabad,Uttar Pradesh,Rural,Standard,26-35,COD,4,Standard,False,False,NaN,4.5,Delivered,1,2015,1,0.20,True,4.10,2015
2,TXN_2015_00000003,2015-01-24,CUST_2015_00004782,PROD_000039,Samsung Galaxy Note 5 64GB Black,Electronics,Smartphones,Samsung,97644.25,46.93,"51,818.26",2,"103,636.52",NaN,"103,636.52",Mumbai,Maharashtra,Metro,Premium,26-35,COD,4,Standard,False,True,Republic Day Sale,NaN,Delivered,1,2015,1,0.17,True,3.30,2015


In [12]:
products_catalog = pd.read_csv(os.path.join(RAW_DIR, "amazon_india_products_catalog.csv"))

products_catalog.info()

print("Duplicates:", products_catalog.duplicated(subset='product_id').sum())

products_catalog.columns = products_catalog.columns.str.lower()

<class 'pandas.DataFrame'>
RangeIndex: 2004 entries, 0 to 2003
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         2004 non-null   str    
 1   product_name       2004 non-null   str    
 2   category           2004 non-null   str    
 3   subcategory        2004 non-null   str    
 4   brand              2004 non-null   str    
 5   base_price_2015    2004 non-null   float64
 6   weight_kg          2004 non-null   float64
 7   rating             2004 non-null   float64
 8   is_prime_eligible  2004 non-null   bool   
 9   launch_year        2004 non-null   int64  
 10  model              2004 non-null   str    
dtypes: bool(1), float64(3), int64(1), str(6)
memory usage: 158.6 KB
Duplicates: 0


In [13]:
# Standardize column names (you already did)
products_catalog.columns = products_catalog.columns.str.lower()

# Remove extra spaces (important for joins!)
products_catalog['product_id'] = products_catalog['product_id'].str.strip()
products_catalog['category'] = products_catalog['category'].str.strip()
products_catalog['brand'] = products_catalog['brand'].str.strip()

# Optional: lowercase for consistency
products_catalog['category'] = products_catalog['category'].str.lower()
products_catalog['brand'] = products_catalog['brand'].str.lower()

In [14]:
df_raw['product_id'] = df_raw['product_id'].str.strip()

In [15]:
products_catalog.head(5)

,product_id,product_name,category,subcategory,brand,base_price_2015,weight_kg,rating,is_prime_eligible,launch_year,model
0,PROD_000001,Apple iPhone 6 16GB Black,electronics,Smartphones,apple,"190,469.10",0.21,3.90,True,2015,iPhone 6
1,PROD_000002,Apple iPhone 6 32GB Black,electronics,Smartphones,apple,"158,424.89",0.18,3.40,True,2015,iPhone 6
2,PROD_000003,Apple iPhone 6 64GB Black,electronics,Smartphones,apple,"118,141.16",0.22,4.60,True,2015,iPhone 6
3,PROD_000004,Apple iPhone 6 16GB White,electronics,Smartphones,apple,"211,721.16",0.21,4.10,True,2015,iPhone 6
4,PROD_000005,Apple iPhone 6 32GB White,electronics,Smartphones,apple,"114,806.24",0.20,3.80,True,2015,iPhone 6


In [17]:
merged_df = df_raw.merge(products_catalog, on='product_id', how='left')
print(f"Merged shape: {merged_df.shape[0]:,} rows × {merged_df.shape[1]} columns")
merged_df.head(3)

Merged shape: 1,127,609 rows × 45 columns


,transaction_id,order_date,customer_id,product_id,product_name_x,category_x,subcategory_x,brand_x,original_price_inr,discount_percent,discounted_price_inr,quantity,subtotal_inr,delivery_charges,final_amount_inr,customer_city,customer_state,customer_tier,customer_spending_tier,customer_age_group,payment_method,delivery_days,delivery_type,is_prime_member,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible_x,product_rating,source_year,product_name_y,category_y,subcategory_y,brand_y,base_price_2015,weight_kg,rating,is_prime_eligible_y,launch_year,model
0,TXN_2015_00000001,2015-01-25,CUST_2015_00003884,PROD_000021,Samsung Galaxy S6 16GB Black,Electronics,Smartphones,Samsung,123614.29,27.91,"89,112.17",1,"89,112.17",0.00,"89,112.17",Mumbai,Maharashtra,Metro,Premium,46-55,COD,6,Standard,No,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.19,True,4.70,2015,Samsung Galaxy S6 16GB Black,electronics,Smartphones,samsung,"123,614.29",0.19,4.70,True,2015,Galaxy S6
1,TXN_2015_00000002,2015-01-05,CUST_2015_00011709,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,"54,731.86",1,"54,731.86",0.00,"54,731.86",Allahabad,Uttar Pradesh,Rural,Standard,26-35,COD,4,Standard,False,False,NaN,4.5,Delivered,1,2015,1,0.20,True,4.10,2015,OnePlus OnePlus 2 16GB White,electronics,Smartphones,oneplus,"54,731.86",0.20,4.10,True,2015,OnePlus 2
2,TXN_2015_00000003,2015-01-24,CUST_2015_00004782,PROD_000039,Samsung Galaxy Note 5 64GB Black,Electronics,Smartphones,Samsung,97644.25,46.93,"51,818.26",2,"103,636.52",NaN,"103,636.52",Mumbai,Maharashtra,Metro,Premium,26-35,COD,4,Standard,False,True,Republic Day Sale,NaN,Delivered,1,2015,1,0.17,True,3.30,2015,Samsung Galaxy Note 5 64GB Black,electronics,Smartphones,samsung,"97,644.25",0.17,3.30,True,2015,Galaxy Note 5


In [18]:
before = len(merged_df)
df     = merged_df.drop_duplicates(subset=['transaction_id'], keep='first')
after  = len(df)
print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}")
print(f"Removed     : {before - after:,} duplicate rows")

Rows before : 1,127,609
Rows after  : 1,127,609
Removed     : 0 duplicate rows


In [20]:
def parse_any_date(val):
    if pd.isna(val): return pd.NaT
    val = str(val).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y"):
        try: return pd.to_datetime(val, format=fmt)
        except: pass
    try: return pd.to_datetime(val, dayfirst=True, errors='coerce')
    except: return pd.NaT

print("Parsing dates... (may take ~60s on large dataset)")
df['order_date'] = df['order_date'].apply(parse_any_date)
invalid = df['order_date'].isna().sum()
df = df.dropna(subset=['order_date'])
#df['order_date']    = df['order_date'].dt.strftime('%Y-%m-%d')
df['order_year']    = pd.to_datetime(df['order_date']).dt.year
df['order_month']   = pd.to_datetime(df['order_date']).dt.month
df['order_quarter'] = pd.to_datetime(df['order_date']).dt.quarter

print(f"✅ Dates cleaned | Invalid removed: {invalid:,}")
print(f"   Range: {df['order_date'].min()} → {df['order_date'].max()}")

Parsing dates... (may take ~60s on large dataset)
✅ Dates cleaned | Invalid removed: 0
   Range: 2015-01-01 00:00:00 → 2025-12-31 00:00:00


In [21]:
PRICE_COLS = [c for c in ['original_price_inr','discounted_price_inr',
                           'subtotal_inr','final_amount_inr','delivery_charges']
              if c in df.columns]

def clean_price(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    if val.lower() in ('price on request','free','n/a','','nan','none','-'): return np.nan
    val = val.replace('₹','').replace('$','').replace(',','').replace(' ','')
    try: return float(val)
    except: return np.nan

for col in PRICE_COLS:
    df[col] = df[col].apply(clean_price)

# Fix 100× outliers (misplaced decimal)
for col in ['original_price_inr', 'final_amount_inr']:
    if col not in df.columns: continue
    median_val = df[col].median()
    mask = df[col] > (median_val * 100)
    df.loc[mask, col] = df.loc[mask, col] / 100
    print(f"  {col}: corrected {mask.sum():,} outliers")

print("\n✅ Price columns cleaned")

  original_price_inr: corrected 1,288 outliers
  final_amount_inr: corrected 0 outliers

✅ Price columns cleaned


In [22]:
def parse_rating(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    m = re.match(r'^(\d+\.?\d*)\s*stars?$', val)
    if m: return float(m.group(1))
    m = re.match(r'^(\d+\.?\d*)\s*/\s*(\d+\.?\d*)$', val)
    if m:
        n, d = float(m.group(1)), float(m.group(2))
        return round((n / d) * 5.0, 1) if d > 0 else np.nan
    try:
        r = float(val)
        return r if 1.0 <= r <= 5.0 else np.nan
    except: return np.nan

for col in ['customer_rating', 'product_rating']:
    if col not in df.columns: continue
    df[col] = df[col].apply(parse_rating)
    med     = df[col].median()
    df[col] = df[col].fillna(med).clip(1.0, 5.0)
    print(f"  {col}: median={med:.2f}, nulls filled")

print("\n✅ Ratings standardised to 1.0–5.0")

  customer_rating: median=4.50, nulls filled
  product_rating: median=4.00, nulls filled

✅ Ratings standardised to 1.0–5.0


In [23]:
CITY_MAP = {
    'bangalore':'Bengaluru', 'bengaluru':'Bengaluru', 'banglore':'Bengaluru',
    'bombay':'Mumbai',       'mumbai':'Mumbai',
    'delhi':'New Delhi',     'new delhi':'New Delhi',
    'madras':'Chennai',      'chennai':'Chennai',
    'calcutta':'Kolkata',    'kolkata':'Kolkata',
    'hydrabad':'Hyderabad',  'hyderabad':'Hyderabad',
    'allahabad':'Prayagraj', 'prayagraj':'Prayagraj',
    'poona':'Pune',          'pune':'Pune',
    'ahemdabad':'Ahmedabad', 'ahmedabad':'Ahmedabad',
    'cochin':'Kochi',        'kochi':'Kochi',
    'mysore':'Mysuru',       'mysuru':'Mysuru',
}

def std_city(val):
    if pd.isna(val): return 'Unknown'
    key = str(val).strip().lower()
    return CITY_MAP.get(key, str(val).strip().title())

if 'customer_city' in df.columns:
    b = df['customer_city'].nunique()
    df['customer_city'] = df['customer_city'].apply(std_city)
    a = df['customer_city'].nunique()
    print(f"  Cities: {b} → {a} unique (merged {b-a} variants)")

print("\n✅ City names standardised")

  Cities: 50 → 34 unique (merged 16 variants)

✅ City names standardised


In [24]:
BOOL_COLS = [c for c in ['is_prime_member','is_prime_eligible','is_festival_sale']
             if c in df.columns]
TRUTHY = {'true','yes','1','y','t','1.0'}

def to_bin(val):
    if pd.isna(val): return 0
    return 1 if str(val).strip().lower() in TRUTHY else 0

for col in BOOL_COLS:
    df[col] = df[col].apply(to_bin)
    print(f"  {col}: {df[col].sum():,} True | {(df[col]==0).sum():,} False")

print("\n✅ Boolean columns → 0/1")

  is_prime_member: 429,124 True | 698,485 False
  is_festival_sale: 349,873 True | 777,736 False

✅ Boolean columns → 0/1


In [25]:
CAT_MAP = {
    'electronics':'Electronics', 'electronic':'Electronics',
    'electronics & accessories':'Electronics',
    'fashion':'Fashion', 'clothing':'Fashion', 'apparel':'Fashion',
    'books':'Books', 'books & education':'Books',
    'home & kitchen':'Home & Kitchen', 'home':'Home & Kitchen',
    'sports':'Sports & Fitness', 'sports & fitness':'Sports & Fitness',
    'health':'Health & Beauty', 'beauty':'Health & Beauty',
    'health & beauty':'Health & Beauty',
    'toys':'Toys & Games', 'toys & games':'Toys & Games',
    'automotive':'Automotive',
}

def std_cat(val):
    if pd.isna(val): return 'Other'
    return CAT_MAP.get(str(val).strip().lower(), str(val).strip().title())

for col in ['category','subcategory']:
    if col in df.columns:
        b = df[col].nunique()
        df[col] = df[col].apply(std_cat)
        a = df[col].nunique()
        print(f"  {col}: {b} → {a} unique values")

print("\n✅ Categories standardised")


✅ Categories standardised


In [26]:
def parse_delivery(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    if 'same' in val: return 0
    m = re.match(r'^(\d+)', val)
    if m:
        v = int(m.group(1))
        return v if 0 <= v <= 30 else np.nan
    try:
        v = float(val)
        return v if 0 <= v <= 30 else np.nan
    except: return np.nan

if 'delivery_days' in df.columns:
    df['delivery_days'] = df['delivery_days'].apply(parse_delivery)
    med = df['delivery_days'].median()
    df['delivery_days'] = df['delivery_days'].fillna(med)
    print(f"  Range: {df['delivery_days'].min():.0f}–{df['delivery_days'].max():.0f} days | Median: {med:.1f}")

print("\n✅ delivery_days cleaned")

  Range: 0–15 days | Median: 3.0

✅ delivery_days cleaned


In [27]:
PAY_MAP = {
    'upi':'UPI','phonepe':'UPI','googlepay':'UPI','gpay':'UPI','paytm':'UPI','bhim':'UPI',
    'cod':'COD','cash on delivery':'COD','c.o.d':'COD','cash':'COD',
    'credit card':'Credit Card','cc':'Credit Card','credit_card':'Credit Card',
    'debit card':'Debit Card','dc':'Debit Card','debit_card':'Debit Card',
    'net banking':'Net Banking','netbanking':'Net Banking',
    'emi':'EMI','no cost emi':'EMI',
    'bnpl':'BNPL','buy now pay later':'BNPL',
    'amazon pay':'Amazon Pay','amazonpay':'Amazon Pay',
    'wallet':'Wallet',
}

def std_pay(val):
    if pd.isna(val): return 'Other'
    return PAY_MAP.get(str(val).strip().lower(), str(val).strip().title())

if 'payment_method' in df.columns:
    df['payment_method'] = df['payment_method'].apply(std_pay)
    print(df['payment_method'].value_counts().to_string())

print("\n✅ Payment methods standardised")

payment_method
UPI            384228
COD            322831
Credit Card    172261
Debit Card     140202
Net Banking     64971
Wallet          22821
BNPL            20295

✅ Payment methods standardised


In [28]:
# Text columns
for col in ['festival_name','customer_state','brand','delivery_type','return_status']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Numeric columns
if 'discount_percent' in df.columns:
    df['discount_percent'] = df['discount_percent'].fillna(0)
if 'quantity' in df.columns:
    df['quantity'] = df['quantity'].fillna(1).clip(lower=1)

df['order_date'] = pd.to_datetime(df['order_date'])

remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print("✅ No nulls remaining!")
else:
    print("Remaining nulls:")
    print(remaining.to_string())

print(f"\n✅ Final dataset: {len(df):,} rows × {df.shape[1]} columns")

Remaining nulls:
original_price_inr     33630
delivery_charges       90201
customer_age_group    135315

✅ Final dataset: 1,127,609 rows × 45 columns


In [29]:
df['original_price_inr'] = df['original_price_inr'].fillna(df['discounted_price_inr'])

df['delivery_charges'] = df['delivery_charges'].fillna(0)

df['customer_age_group'] = df['customer_age_group'].fillna('Unknown')

In [30]:
df.isnull().sum()

transaction_id            0
order_date                0
customer_id               0
product_id                0
product_name_x            0
category_x                0
subcategory_x             0
brand_x                   0
original_price_inr        0
discount_percent          0
discounted_price_inr      0
quantity                  0
subtotal_inr              0
delivery_charges          0
final_amount_inr          0
customer_city             0
customer_state            0
customer_tier             0
customer_spending_tier    0
customer_age_group        0
payment_method            0
delivery_days             0
delivery_type             0
is_prime_member           0
is_festival_sale          0
festival_name             0
customer_rating           0
return_status             0
order_month               0
order_year                0
order_quarter             0
product_weight_kg         0
is_prime_eligible_x       0
product_rating            0
source_year               0
product_name_y      

In [31]:
original_rows = sum(
    len(pd.read_csv(f, low_memory=False)) for f in year_files
)

print("=" * 55)
print("DATA QUALITY REPORT")
print("=" * 55)
print(f"""
Source files   : {len(year_files)} year files (2015–2025)
Original rows  : {original_rows:,}
Clean rows     : {len(df):,}
Rows removed   : {original_rows - len(df):,}

Columns cleaned:
  ✅ order_date       Standardised to YYYY-MM-DD
  ✅ Price columns    Stripped ₹/commas, fixed 100× outliers
  ✅ customer_rating  Normalised to 1.0–5.0 scale
  ✅ product_rating   Normalised to 1.0–5.0 scale
  ✅ customer_city    Resolved city name variants
  ✅ is_prime_member  Converted to 0/1
  ✅ is_festival_sale Converted to 0/1
  ✅ category         Resolved casing/naming variants
  ✅ delivery_days    Removed invalid values, filled nulls
  ✅ payment_method   Consolidated to 9 clean categories

Date range     : {df['order_date'].min().date()} → {df['order_date'].max().date()}
""")

DATA QUALITY REPORT

Source files   : 11 year files (2015–2025)
Original rows  : 1,127,609
Clean rows     : 1,127,609
Rows removed   : 0

Columns cleaned:
  ✅ order_date       Standardised to YYYY-MM-DD
  ✅ Price columns    Stripped ₹/commas, fixed 100× outliers
  ✅ customer_rating  Normalised to 1.0–5.0 scale
  ✅ product_rating   Normalised to 1.0–5.0 scale
  ✅ customer_city    Resolved city name variants
  ✅ is_prime_member  Converted to 0/1
  ✅ is_festival_sale Converted to 0/1
  ✅ category         Resolved casing/naming variants
  ✅ delivery_days    Removed invalid values, filled nulls
  ✅ payment_method   Consolidated to 9 clean categories

Date range     : 2015-01-01 → 2025-12-31



In [32]:
# Full dataset
main_path = os.path.join(EXPORTS_DIR, "amazon_transactions_clean.csv")
df.to_csv(main_path, index=False)
size_mb = os.path.getsize(main_path) / 1e6
print(f"✅ Full export   → {main_path}")
print(f"   File size     : {size_mb:.1f} MB")

# Sample (50K rows — use this while BUILDING Power BI, switch to full when done)
sample_path = os.path.join(EXPORTS_DIR, "amazon_transactions_sample.csv")
df.sample(min(50000, len(df)), random_state=42).to_csv(sample_path, index=False)
print(f"✅ Sample export → {sample_path}")

# Products catalog
catalog_raw = os.path.join(RAW_DIR, "amazon_india_products_catalog.csv")
if os.path.exists(catalog_raw):
    cat_df = pd.read_csv(catalog_raw)
    for col in ['category','subcategory']:
        if col in cat_df.columns:
            cat_df[col] = cat_df[col].apply(std_cat)
    cat_path = os.path.join(EXPORTS_DIR, "amazon_products_catalog_clean.csv")
    cat_df.to_csv(cat_path, index=False)
    print(f"✅ Catalog export → {cat_path}")

# Per-year files
for yr in sorted(df['order_year'].unique()):
    yr_path = os.path.join(CLEANED_DIR, f"amazon_{yr}_clean.csv")
    df[df['order_year']==yr].to_csv(yr_path, index=False)
print(f"✅ Per-year files → {CLEANED_DIR}/")

print("\n" + "="*55)
print("PHASE 1 COMPLETE ✅")
print("="*55)
print("""
Next steps:
  1. Check exports/ folder — confirm CSVs are there
  2. Run: git add . && git commit -m "Phase 1: Data cleaning complete"
  3. Open 02_eda.ipynb for Phase 2
""")

✅ Full export   → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\exports\amazon_transactions_clean.csv
   File size     : 426.0 MB
✅ Sample export → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\exports\amazon_transactions_sample.csv
✅ Catalog export → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\exports\amazon_products_catalog_clean.csv
✅ Per-year files → C:/Users/pooja/OneDrive/Desktop/Amazon_Sales_Analytics\data\cleaned/

PHASE 1 COMPLETE ✅

Next steps:
  1. Check exports/ folder — confirm CSVs are there
  2. Run: git add . && git commit -m "Phase 1: Data cleaning complete"
  3. Open 02_eda.ipynb for Phase 2

